# GPT-2 scale parity

Choose one pretrained GPT-2 size and compare with the scratch implementation. Run one size at a time. 1.5B check would be faster in Colab, but I am running locally.

In [1]:
import gc
import os
import sys
from pathlib import Path

os.environ['HF_HUB_DISABLE_XET'] = '1'

root = Path.cwd()
candidates = (root, root.parent,
              root / '002_GPT2_From_Scratch', root.parent / '002_GPT2_From_Scratch',
              root / '003_My GPT From Scratch', root.parent / '003_My GPT From Scratch')
package_root = next((path for path in candidates if (path / 'gpt2_scratch').is_dir()), None)
if package_root is None:
    raise FileNotFoundError('scratch package folder not found')
sys.path.insert(0, str(package_root))

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from gpt2_scratch import ScratchGPT2Config, ScratchGPT2ForCausalLM, load_weights_into_gpt
from gpt2_scratch.weights import reference_checkpoint

In [ ]:
# choose gpt2, gpt2-medium, gpt2-large, gpt2-xl
#MODEL_NAME = 'gpt2'
MODEL_NAME = 'gpt2-xl'

model_id, revision = reference_checkpoint(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'model': MODEL_NAME, 'checkpoint': model_id, 'revision': revision, 'device': str(device)})

{'model': 'gpt2-xl', 'checkpoint': 'openai-community/gpt2-xl', 'revision': '15ea56dee5df4983c59b2538573817e1667135e2', 'device': 'cpu'}


In [3]:
if 'reference' in globals():
    del reference
if 'scratch' in globals():
    del scratch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = GPT2TokenizerFast.from_pretrained(model_id, revision=revision)
tokenizer.pad_token = tokenizer.eos_token
reference = GPT2LMHeadModel.from_pretrained(
    model_id, revision=revision, dtype=torch.float32
).to(device).eval()
scratch = ScratchGPT2ForCausalLM(
    ScratchGPT2Config.from_scale(MODEL_NAME)
).to(device).eval()
coverage = load_weights_into_gpt(scratch, reference.state_dict())

model.safetensors:   6%|6         | 402M/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
parameter_count = sum(parameter.numel() for parameter in scratch.parameters())
tied = scratch.out_head.weight is scratch.tok_emb.weight
print({'model': MODEL_NAME, 'parameters': parameter_count, 'tied': tied,
       'source_keys': len(coverage['accounted_source_keys'])})

{'model': 'gpt2-xl', 'parameters': 1557611200, 'tied': True, 'source_keys': 581}


In [5]:
inputs = tokenizer(['Hello, I am a language model.', 'The answer is'], return_tensors='pt', padding=True)
inputs = {name: value.to(device) for name, value in inputs.items()}
position_ids = inputs['attention_mask'].cumsum(-1) - 1
position_ids.masked_fill_(inputs['attention_mask'] == 0, 1)
with torch.no_grad():
    reference_logits = reference(**inputs, position_ids=position_ids).logits
    scratch_logits = scratch(**inputs, use_cache=False).logits
valid = inputs['attention_mask'].bool().unsqueeze(-1).expand_as(scratch_logits)
max_logit_error = (reference_logits[valid] - scratch_logits[valid]).abs().max().item()
print('max logit error:', max_logit_error)

max logit error: 0.0


In [6]:
prompt = tokenizer('Hello, I am a language model.', return_tensors='pt')
prompt = {name: value.to(device) for name, value in prompt.items()}
reference_tokens = reference.generate(**prompt, max_new_tokens=32, do_sample=False, pad_token_id=50256)
scratch_tokens = scratch.generate(**prompt, max_new_tokens=32, do_sample=False, pad_token_id=50256)
assert torch.equal(reference_tokens, scratch_tokens)
print(tokenizer.decode(scratch_tokens[0]))

Hello, I am a language model. I am a language model that can be used to model any language. I can be used to model any language that has a grammar. I can be used to
